# DiT and U-Net setups

Compare the two configured architectures on the same illustrative rendered pair.
Both concatenate the evolving image and coarse condition into six input channels
and predict three velocity channels. Both use the same deterministic rectified flow.

This notebook initializes fresh models and performs **one optimizer step per
architecture**. It checks execution and memory, not learned zoom quality or an
architecture ranking. It writes no checkpoints and starts no long training run.

Prerequisites: CUDA, repository dependencies. Run all cells in order.

In [1]:
from pathlib import Path
import sys
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "model.py").exists() and (p / "mandelbrot.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
assert torch.cuda.is_available(), "These notebooks use the CUDA renderer."
torch.manual_seed(20261016)
print(torch.cuda.get_device_name())
%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 11})

NVIDIA GeForce RTX 4090


In [2]:
from mandelbrot import ColorRenderer, View

SIZE = 128
AA = 8  # 8x8 subpixel samples (64 per pixel), averaged after coloring.
renderer = ColorRenderer(SIZE * AA, SIZE * AA)

def render_rgb(view):
    # ColorRenderer reuses buffers: clone before retaining a render.
    rgb = renderer.render(view).clone().permute(2, 0, 1)[None]
    return F.avg_pool2d(rgb, AA)

def zoom_pair(parent_view, left, top):
    side = SIZE // 4
    assert 0 <= left <= SIZE-side and 0 <= top <= SIZE-side
    parent = render_rgb(parent_view)
    crop = parent[:, :, top:top+side, left:left+side]
    coarse = F.interpolate(crop, size=(SIZE, SIZE), mode="bilinear", align_corners=False)
    # Positive image y points down; positive imaginary coordinates point up.
    offset = complex(left + side/2 - SIZE/2, SIZE/2 - top - side/2)
    center = parent_view.center + offset * parent_view.span / SIZE
    child_view = View(center=center, span=parent_view.span / 4,
                      iterations=parent_view.iterations)
    target = render_rgb(child_view)
    return parent, coarse, target, child_view

def show_rgb(axis, image, title):
    axis.imshow(image[0].detach().float().cpu().permute(1, 2, 0).clamp(0, 1), interpolation="nearest")
    axis.set_title(title)
    axis.axis("off")

Warp 1.18.0 initialized:
   CUDA Toolkit 13.4, Driver 13.0
   Devices:
     "cpu"      : "CPU"
     "cuda:0"   : "NVIDIA GeForce RTX 4090" (24 GiB, sm_89, mempool enabled)
   Kernel cache:
     /home/blacknoir/.cache/warp/1.18.0


In [3]:
import json
import gc
import copy
import time
from model import build_model, RectifiedFlow

configs = {name: json.loads((ROOT / "configs" / f"zoom4x-128-{name}.json").read_text())
           for name in ("dit", "unet")}
for name, config in configs.items():
    print(f"{name}: {config['model']}")
shared = ("formulation", "data", "size", "batch_size", "learning_rate", "ema_decay", "seed")
assert all(configs['dit'][key] == configs['unet'][key] for key in shared)

dit: {'channels': 3, 'condition_channels': 3, 'patch_size': 4, 'hidden_size': 256, 'depth': 8, 'num_heads': 8, 'mlp_ratio': 4.0}
unet: {'channels': 3, 'widths': [64, 128, 256, 256]}


## A real pair, a small execution batch

Use batch 2 here to keep reruns quick; the training configs use batch 32.
The batch repeats one pair, so it is not a meaningful generalization experiment.
DiT has 1024 tokens (128/4 squared); U-Net attention is restricted to its two
lowest spatial resolutions. Parameter counts and computational budgets differ.

In [4]:
_, coarse, target, _ = zoom_pair(
    View(center=-.7435+.1314j, span=.012, iterations=8192), 48, 48)
y = (coarse*2-1).repeat(2, 1, 1, 1)
x = (target*2-1).repeat(2, 1, 1, 1)
fixed_t = torch.tensor([.25, .75], device="cuda")
flow = RectifiedFlow()

Module mandelbrot 617f2d0 load on device 'cuda:0' took 0.31 ms  (cached)


In [5]:
results = []
for name, config in configs.items():
    torch.manual_seed(config["seed"])
    torch.cuda.empty_cache()
    model = build_model(config).cuda()
    ema = copy.deepcopy(model).eval().requires_grad_(False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"], weight_decay=0)
    torch.cuda.reset_peak_memory_stats()
    baseline_bytes = torch.cuda.memory_allocated()
    torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.autocast("cuda", dtype=torch.bfloat16):
        loss = flow.loss(model, x, y, fixed_t)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)
    assert torch.isfinite(loss) and torch.isfinite(grad_norm)
    optimizer.step()
    with torch.no_grad():
        for destination, source in zip(ema.parameters(), model.parameters()):
            destination.lerp_(source, 1-config["ema_decay"])
    torch.cuda.synchronize()
    elapsed = time.perf_counter()-started
    peak = torch.cuda.max_memory_allocated()
    with torch.autocast("cuda", dtype=torch.bfloat16):
        prediction = flow.sample(ema, y[:1], steps=4)
    assert prediction.shape == target.shape and prediction.isfinite().all()
    results.append(dict(model=name, parameters=sum(p.numel() for p in model.parameters()),
                        velocity_mse=loss.item(), peak_gib=peak/2**30,
                        extra_peak_gib=(peak-baseline_bytes)/2**30,
                        first_step_seconds=elapsed))
    del model, ema, optimizer, loss, prediction, destination, source
    gc.collect()

print(f"{'Model':<8} {'Parameters':>12} {'Velocity MSE':>14} {'Peak GiB':>10} {'Extra GiB':>10} {'First step s':>13}")
for row in results:
    print(f"{row['model']:<8} {row['parameters']:>12,} {row['velocity_mse']:>14.5f} "
          f"{row['peak_gib']:>10.3f} {row['extra_peak_gib']:>10.3f} {row['first_step_seconds']:>13.3f}")

Model      Parameters   Velocity MSE   Peak GiB  Extra GiB  First step s
dit         9,768,240        0.02847      0.342      0.246         0.243
unet       14,856,835        0.02847      0.641      0.488         0.129


## Interpret the check

The loss is measured before the first optimizer step; both output heads start at
zero. Similar initial losses are expected, not evidence that the models learn
equally well. Memory includes model, EMA, AdamW and activations plus the tiny
rendering workload; it excludes the full dataset. The first-step time includes
startup overhead and is not steady-state throughput.

The actual paired dataset has not been generated. Once available, use:

```sh
uv run python train.py --config configs/zoom4x-128-dit.json
uv run python train.py --config configs/zoom4x-128-unet.json
```

Compare held-out branches and recursive 4x/16x/64x/256x trajectories against
fresh renders. Keep data and evaluation identical; these defaults are not
parameter-matched comparisons.

**Optional experiment:** change the local batch from 2 to 4 and rerun the final
code cell to measure the memory effect. This does not modify either config.